<a href="https://colab.research.google.com/github/mhtrannn/Dynamic-Customer-Segmentation-/blob/main/Step%205_Dynamic%20profiles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from dateutil.relativedelta import relativedelta
import numpy as np

# 1. DATA PREPROCESSING & WINDOW GENERATION
# Load data and format dates
df = pd.read_csv("online_retail_cleaned CusID.csv")
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Filter data from Dec 2009 to Nov 2011 (using < 2011-12-01 to capture all of Nov 30)
start_period = pd.to_datetime('2009-12-01')
end_period = pd.to_datetime('2011-12-01')
df = df[(df['InvoiceDate'] >= start_period) & (df['InvoiceDate'] < end_period)]

# Generate 22 overlapping 3-month windows (sliding by 1 month)
windows = []
current_start = pd.to_datetime('2009-12-01')
while current_start < pd.to_datetime('2011-10-01'):
    current_end = current_start + relativedelta(months=3)
    windows.append({'start': current_start, 'end': current_end})
    current_start += relativedelta(months=1)

# Identify the acquisition date (first purchase) for each customer
first_purchase = df.groupby('Customer ID')['InvoiceDate'].min().reset_index()
first_purchase.rename(columns={'InvoiceDate': 'First_Date'}, inplace=True)

# 2. GLOBAL PRE-CALCULATION (INTERVAL & TREND)
# Aggregate revenue at the invoice level
invoices = df.groupby(['Customer ID', 'Invoice', 'InvoiceDate']).agg(
    Revenue=('Revenue', 'sum')
).reset_index().sort_values(['Customer ID', 'InvoiceDate'])

# Retrieve the date and revenue of the immediately preceding invoice
invoices['Prev_InvoiceDate'] = invoices.groupby('Customer ID')['InvoiceDate'].shift(1)
invoices['Prev_Revenue'] = invoices.groupby('Customer ID')['Revenue'].shift(1)

# Calculate Purchase Interval (elapsed days between consecutive invoices)
invoices['Interval'] = (invoices['InvoiceDate'] - invoices['Prev_InvoiceDate']).dt.days

# Calculate Spending Trend (relative financial variance between consecutive invoices)
invoices['Spending_Trend'] = np.where(
    (invoices['Prev_Revenue'].notna()) & (invoices['Prev_Revenue'] > 0),
    (invoices['Revenue'] - invoices['Prev_Revenue']) / invoices['Prev_Revenue'],
    0
)

# 3. FEATURE EXTRACTION PER WINDOW
all_profiles = []
for i, w in enumerate(windows):
    w_start, w_end = w['start'], w['end']

    # Keep only active customers acquired prior to the end of the current window
    active_customers = first_purchase[first_purchase['First_Date'] < w_end]['Customer ID'].unique()
    base = pd.DataFrame({'Customer ID': active_customers})

    # Calculate Recency (strictly historical to prevent data leakage)
    df_history = df[df['InvoiceDate'] < w_end]
    recency_df = df_history.groupby('Customer ID').agg(Last_Purchase=('InvoiceDate', 'max')).reset_index()
    recency_df['Recency'] = (w_end - recency_df['Last_Purchase']).dt.days
    base = pd.merge(base, recency_df[['Customer ID', 'Recency']], on='Customer ID', how='left')

    # Calculate Frequency, Monetary, and Product Diversity (intra-window)
    df_w = df[(df['InvoiceDate'] >= w_start) & (df['InvoiceDate'] < w_end)]
    rfm = df_w.groupby('Customer ID').agg(
        Frequency=('Invoice', 'nunique'),
        Monetary=('Revenue', 'sum'),
        Product_Diversity=('StockCode', 'nunique')
    ).reset_index()
    base = pd.merge(base, rfm, on='Customer ID', how='left')

    # Merge pre-calculated Interval and Trend into the current window
    inv_w = invoices[(invoices['InvoiceDate'] >= w_start) & (invoices['InvoiceDate'] < w_end)]
    inv_agg = inv_w.groupby('Customer ID').agg(
        Purchase_Interval=('Interval', 'mean'),      # Mean of true intervals within the window
        Spending_Trend=('Spending_Trend', 'last')    # Trend of the most recent transaction
    ).reset_index()
    base = pd.merge(base, inv_agg, on='Customer ID', how='left')

    # Apply Panel Data Padding for inactive customers
    base['Frequency'] = base['Frequency'].fillna(0)
    base['Monetary'] = base['Monetary'].fillna(0)
    base['Product_Diversity'] = base['Product_Diversity'].fillna(0)
    base['Purchase_Interval'] = base['Purchase_Interval'].fillna(0)
    base['Spending_Trend'] = base['Spending_Trend'].fillna(0)

    # Calculate Repeat Purchase Ratio and Average Order Value (AOV)
    base['Repeat_Purchase_Ratio'] = np.where(base['Frequency'] > 0, (base['Frequency'] - 1) / base['Frequency'], 0)
    base['AOV'] = np.where(base['Frequency'] > 0, base['Monetary'] / base['Frequency'], 0)

    # Assign Window ID and Number
    base['Window_ID'] = f"W{i+1}"
    base['Window_Num'] = i + 1

    all_profiles.append(base)

# 4. FINAL DATASET COMPILATION & EXPORT
# Concatenate all window profiles and sort chronologically
dynamic_profiles = pd.concat(all_profiles, ignore_index=True)
dynamic_profiles = dynamic_profiles.sort_values(['Customer ID', 'Window_Num'])

# Export the final dataset ready for K-Means clustering
dynamic_profiles.to_csv("dynamic_profiles_step5_FINAL.csv", index=False)
